# Folio — Google Colab Setup

Run each cell in order. The final cell gives you a **public URL** to open Folio in your browser.

> **Note:** Free Colab instances disconnect after ~90 minutes of inactivity. Re-run from Cell 3 (skip install if already done).

## Cell 1 — Install Node.js 20 LTS

Colab ships with an older Node. We install Node 20 via NodeSource.

In [ ]:
%%bash
# Install Node.js 20 LTS from NodeSource
curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - 2>/dev/null
sudo apt-get install -y nodejs 2>/dev/null | tail -3

echo "Node: $(node --version)"
echo "npm:  $(npm --version)"

## Cell 2 — Clone the Repository

In [ ]:
%%bash
# Clone Folio from GitHub
if [ -d "Folio" ]; then
  echo "Already cloned — pulling latest changes"
  cd Folio && git pull origin main 2>&1 | tail -3
else
  git clone https://github.com/azzindani/Folio.git
  echo "Cloned successfully"
fi

# Patch vite.config.ts to allow tunnel hosts (*.loca.lt, *.trycloudflare.com, etc.)
# Uses a heredoc so the patch is always correct regardless of the repo's current config.
cat > /content/Folio/vite.config.ts << 'VITECONF'
import { defineConfig } from 'vite';
import { resolve } from 'path';

export default defineConfig(({ mode }) => ({
  resolve: {
    alias: {
      '@': resolve(__dirname, 'src'),
    },
  },
  define: {
    __DEV__: mode === 'development',
    __PROD__: mode === 'production',
  },
  build: {
    target: 'es2022',
    outDir: 'dist',
    sourcemap: mode !== 'production',
    minify: mode === 'production' ? 'esbuild' : false,
    chunkSizeWarningLimit: 600,
  },
  server: {
    port: 3000,
    open: false,
    strictPort: false,
    allowedHosts: 'all',
  },
  preview: {
    port: 4173,
    allowedHosts: 'all',
  },
}));
VITECONF
echo "vite.config.ts patched — tunnel hosts allowed"

cd Folio && echo "Branch: $(git branch --show-current)" && echo "Commit: $(git log -1 --oneline)"


## Cell 3 — Install Dependencies

Takes ~60–90 seconds on first run. Cached on subsequent runs.

In [ ]:
%%bash
cd Folio

# Use ci for reproducible install from lock file
npm ci --prefer-offline 2>&1 | tail -5

echo ""
echo "Dependencies installed: $(ls node_modules | wc -l) packages"

## Cell 4 — Build Production Bundle

Compiles TypeScript and bundles with Vite. Takes ~15–30 seconds.

In [ ]:
%%bash
cd Folio

npm run build 2>&1 | grep -v "^$" | tail -20

echo ""
echo "Build output:"
ls -lh dist/assets/*.js 2>/dev/null | awk '{print $5, $9}' | sort -rh | head -10

## Cell 5 — Run Type Check & Tests (Optional)

Skip this cell if you just want to launch the editor.

In [ ]:
%%bash
cd Folio

echo "=== TypeScript ==="
npm run typecheck 2>&1

echo ""
echo "=== Unit Tests ==="
npm run test:unit 2>&1 | tail -8

echo ""
echo "=== Integration Tests ==="
npm run test:integration 2>&1 | tail -5

## Cell 6 — Launch Folio + Public Tunnel

Starts the preview server on port 4173 and exposes it via **localtunnel** (no account needed).

After running, click the printed URL to open Folio in your browser.

> The tunnel URL changes each session. Refresh this cell to get a new one.

In [ ]:
import subprocess, threading, time, re

PORT = 4173

# ── Start Vite preview server ──────────────────────────────────
server = subprocess.Popen(
    ['npm', 'run', 'preview', '--', '--host', '0.0.0.0', '--port', str(PORT)],
    cwd='/content/Folio',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Give the server a moment to start
time.sleep(3)
print(f'Preview server started on port {PORT}')

# ── Start localtunnel ─────────────────────────────────────────
tunnel = subprocess.Popen(
    ['npx', 'localtunnel', '--port', str(PORT)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait for the public URL
public_url = None
for _ in range(30):
    line = tunnel.stdout.readline()
    match = re.search(r'https://[a-z0-9\-]+\.loca\.lt', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print(f'\n✅ Folio is live at:\n')
    print(f'  {public_url}')
    print(f'\nOpen that URL in your browser to use Folio.')
    print(f'(You may see a "tunnel password" page — click "Click to continue".)')
else:
    print('Could not get tunnel URL. Try Cell 6b below.')


## Cell 6b — Alternative Tunnel via cloudflared

Use this if Cell 6 localtunnel fails. Cloudflared is more reliable.

In [ ]:
%%bash
# Install cloudflared
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared
echo "cloudflared installed"

In [ ]:
import subprocess, time, re

PORT = 4173

# Make sure server is running (re-start if needed)
subprocess.Popen(
    ['npm', 'run', 'preview', '--', '--host', '0.0.0.0', '--port', str(PORT)],
    cwd='/content/Folio',
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)

# Start cloudflared tunnel
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(60):
    line = proc.stdout.readline()
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print(f'\n✅ Folio is live at:\n')
    print(f'  {public_url}')
    print(f'\nNo password required — opens directly.')
else:
    print('URL not found. Check the cloudflared output above.')


## Cell 7 — Development Mode (Hot Reload)

Use this instead of Cell 6 if you want to make code changes and see them live.
Vite dev server reloads the browser on every file save.

In [ ]:
import subprocess, time, re

PORT = 5173  # Vite dev default

# Start Vite dev server
dev = subprocess.Popen(
    ['npm', 'run', 'dev', '--', '--host', '0.0.0.0', '--port', str(PORT)],
    cwd='/content/Folio',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait for ready
for _ in range(30):
    line = dev.stdout.readline()
    if 'ready' in line.lower() or 'localhost' in line:
        print('Dev server ready')
        break
time.sleep(2)

# Tunnel
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for _ in range(60):
    line = tunnel.stdout.readline()
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        print(f'\n✅ Dev mode URL: {match.group(0)}')
        print('Changes to src/ files reload instantly.')
        break


## Cell 8 — Run All Tests

Runs unit tests + integration tests (no browser needed).

In [ ]:
%%bash
cd Folio

echo "=== Unit Tests (vitest) ==="
npm run test:unit 2>&1

echo ""
echo "=== Integration Tests ==="
npm run test:integration 2>&1

## Cell 9 — Inspect the Design YAML

Folio's designs are plain YAML files. This cell creates a minimal poster YAML and renders it.

In [ ]:
%%bash
cat > /content/my-poster.design.yaml << 'YAML'
_protocol: "design/v1"
_mode: complete

meta:
  id: colab-demo
  name: Colab Demo Poster
  type: poster
  created: "2026-04-10"
  modified: "2026-04-10"
  generator: human

document:
  width: 1080
  height: 1080
  unit: px
  dpi: 96

theme:
  ref: dark-tech

layers:
  - id: bg
    type: rect
    z: 0
    x: 0
    y: 0
    width: 1080
    height: 1080
    fill:
      type: linear
      angle: 135
      stops:
        - { color: "$background", position: 0 }
        - { color: "$surface", position: 100 }

  - id: title
    type: text
    z: 20
    x: 80
    y: 400
    width: 920
    height: auto
    content:
      type: plain
      value: "Hello from Colab!"
    style:
      font_family: "$heading"
      font_size: 80
      font_weight: 800
      color: "$text"
      line_height: 1.1
YAML

echo "Design file created: /content/my-poster.design.yaml"
echo ""
echo "Open Folio, click 'Open' in the file tree, and select this file."
echo "Or drag it into the canvas area."

## Cell 10 — Export Design as SVG (headless)

Uses the Folio Node.js API to export a design without opening a browser.

In [ ]:
%%bash
cd Folio

# Run a quick Node.js script that parses + renders the design to SVG
node --input-type=module << 'JS'
import { readFileSync } from 'fs';

// Use compiled output from dist/
// (For headless use, import directly from src/ with tsx or ts-node)
console.log('Folio headless export — available via MCP server in Phase 2');
console.log('For now, use the browser UI to open and export YAML designs.');
console.log('');
console.log('Installed packages:');

const pkg = JSON.parse(readFileSync('package.json', 'utf-8'));
const deps = Object.keys(pkg.dependencies ?? {});
console.log(deps.join(', '));
JS

## Cell 11 — Stop All Servers

Clean up background processes when you're done.

In [ ]:
%%bash
# Kill preview server, dev server, and tunnel processes
pkill -f 'vite' 2>/dev/null && echo 'Vite stopped' || echo 'Vite was not running'
pkill -f 'localtunnel' 2>/dev/null && echo 'localtunnel stopped' || echo 'localtunnel was not running'
pkill -f 'cloudflared' 2>/dev/null && echo 'cloudflared stopped' || echo 'cloudflared was not running'
echo 'Done'

---

## Quick Reference

| Command | What it does |
|---|---|
| `npm run dev` | Start hot-reload dev server on :5173 |
| `npm run build` | Production build to `dist/` |
| `npm run preview` | Serve the built `dist/` on :4173 |
| `npm run test:unit` | Run 270+ unit tests with Vitest |
| `npm run test:integration` | Run 24 integration tests |
| `npm run test:coverage` | Unit tests + V8 coverage report |
| `npm run lint` | ESLint (zero warnings policy) |
| `npm run typecheck` | TypeScript strict type check |

## Keyboard Shortcuts

| Key | Action |
|---|---|
| `V` | Select tool |
| `R` | Rectangle tool |
| `C` | Circle tool |
| `T` | Text tool |
| `L` | Line tool |
| `G` | Toggle grid |
| `Ctrl+K` or `/` | Command palette |
| `Ctrl+Z` / `Ctrl+Shift+Z` | Undo / Redo |
| `Ctrl+D` | Duplicate selected |
| `Ctrl+C` / `Ctrl+V` | Copy / Paste layers (YAML) |
| `Ctrl+G` | Group selected layers |
| `Ctrl+Shift+G` | Ungroup |
| `Ctrl+[` / `Ctrl+]` | Send backward / Bring forward |
| `Ctrl+0` | Fit canvas to screen |
| `Ctrl+S` | Save file |
| `Ctrl+O` | Open file |
| `Delete` / `Backspace` | Delete selected layers |
| `Escape` | Deselect all |

## Design File Format

Designs are plain YAML stored as `*.design.yaml`:

```yaml
_protocol: "design/v1"
_mode: complete

meta:
  id: my-poster
  name: My Poster
  type: poster          # poster | carousel
  created: "2026-04-10"
  modified: "2026-04-10"
  generator: human      # human | llm

document:
  width: 1080
  height: 1080
  unit: px
  dpi: 96

theme:
  ref: dark-tech        # dark-tech | light-clean | ocean-blue

layers:
  - id: bg
    type: rect
    z: 0                # z-index (0-9 bg, 10-19 structural, 20-49 content)
    x: 0
    y: 0
    width: 1080
    height: 1080
    fill:
      type: solid
      color: "$background"   # $token references the active theme
```